# 04 — Explainable AI and Results

Visualise SHAP values and interpret which radiomic features drive predictions.  
Run `train_ml_models.py` and `explain_models.py` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import joblib
import shap
from pathlib import Path

%matplotlib inline
shap.initjs()

In [ ]:
# Load pre-computed figures from explain_models.py
fig_dir = Path('../results/figures')
for fig_name in ['shap_summary.png', 'shap_bar.png', 'shap_waterfall_p0.png',
                  'roc_curves.png', 'model_comparison_bar.png']:
    fp = fig_dir / fig_name
    if fp.exists():
        img = mpimg.imread(fp)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(fig_name.replace('.png','').replace('_',' ').title())
        plt.tight_layout()
        plt.show()
    else:
        print(f'Not yet generated: {fig_name}')

In [ ]:
# Feature importance by region
imp_path = Path('../results/tables/shap_feature_importance.csv')
if imp_path.exists():
    df_imp = pd.read_csv(imp_path)
    region_summary = df_imp.groupby('Region')['MeanAbsSHAP'].agg(['mean','sum','count'])
    region_summary.columns = ['MeanSHAP', 'TotalSHAP', 'NumFeatures']
    region_summary = region_summary.sort_values('MeanSHAP', ascending=False)
    
    print('\nSHAP Importance by Region:')
    display(region_summary)
    
    # Pie chart
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = ['#d62728','#ff7f0e','#1f77b4','#17becf','#2ca02c','#98df8a','#7f7f7f']
    region_summary['TotalSHAP'].plot.pie(ax=ax, colors=colors[:len(region_summary)],
                                          autopct='%1.1f%%', startangle=90,
                                          pctdistance=0.85)
    ax.set_title('SHAP Importance Distribution by Region')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig('../results/figures/shap_by_region_pie.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nTop 15 features:')
    display(df_imp.head(15)[['Feature','Region','MeanAbsSHAP']])

In [ ]:
# Interactive SHAP from loaded model
model_path = Path('../results/models/best_model.joblib')
ml_path    = Path('../data/features/ml_dataset.csv')

if model_path.exists() and ml_path.exists():
    artifact = joblib.load(model_path)
    df = pd.read_csv(ml_path)
    X  = df[artifact['feature_cols']]
    y  = df[artifact['label_col']].astype(int).values
    
    pipe   = artifact['model']
    scaler = pipe.named_steps.get('scaler')
    clf    = pipe.named_steps['clf']
    
    X_sc = pd.DataFrame(scaler.transform(X) if scaler else X, columns=X.columns)
    
    try:
        explainer   = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X_sc)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    except Exception:
        bg = shap.sample(X_sc, 30)
        explainer   = shap.KernelExplainer(lambda x: clf.predict_proba(x)[:,1], bg)
        shap_values = explainer.shap_values(X_sc, nsamples=50)
    
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sc, max_display=20, show=False)
    plt.title('Interactive SHAP Summary')
    plt.tight_layout()
    plt.show()
else:
    print('Run train_ml_models.py and explain_models.py first.')